# Calibrating two uncertain sensor axes
Two instruments observe the same latent calibration settings. Estimate their linear relation when either reading may be in error. Watch the fitted line rotate as you change the stated measurement uncertainties. Noise draws are fixed for repeatable comparisons.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

sigma_x = .45                       # mV
sigma_y = .40                       # K
assert .08 <= sigma_x <= 1.5 and .08 <= sigma_y <= 1.5
x = np.array([-2.35,-1.02,-1.12,.57,.42,1.36,2.27])
y = np.array([-3.53,-2.71,-1.15,.21,1.23,2.54,3.89])
P = np.column_stack((x,y))
mean = P.mean(axis=0)

def orthogonal_fit(points, uncertainties):
    center = points.mean(axis=0)
    Z = (points-center)/uncertainties
    _,singular,vt = np.linalg.svd(Z,full_matrices=False)
    normal = vt[-1]
    projected = Z - np.outer(Z@normal,normal)
    direction = vt[0]*uncertainties
    return center,direction,center+projected*uncertainties,singular[-1]**2

scales = np.array([sigma_x,sigma_y])
center,direction,foot,loss = orthogonal_fit(P,scales)
slope = direction[1]/direction[0]
intercept = center[1]-slope*center[0]
ols = np.polyfit(x,y,1)
print(f"Weighted orthogonal slope {slope:.6f} K/mV, intercept {intercept:.6f} K, loss {loss:.6f}")
print(f"Vertical least-squares slope {ols[0]:.6f} K/mV")

## Eliminate a different nuisance coordinate for every reading
For a trial slope through the centroid, minimising the squared correction over the unknown true horizontal coordinate leaves a single scalar loss. Plot this independent function and compare its minimum with the singular-vector fit.

In [ ]:
def profile(b):
    return np.sum(((y-mean[1])-b*(x-mean[0]))**2)/(sigma_y**2+b*b*sigma_x**2)
assert np.isclose(loss,profile(slope),rtol=2e-12)
assert np.isclose(loss,np.sum(((P-foot)/scales)**2),rtol=2e-12)
fig, (data, cost) = plt.subplots(1,2,figsize=(10,4),layout="constrained")
data.scatter(x,y,color="#145c71",label="readings",zorder=4)
for i,(observation,adjusted) in enumerate(zip(P,foot)):
    data.plot([observation[0],adjusted[0]],[observation[1],adjusted[1]],color="#9e4429",lw=2,label="adjustments" if i==0 else None)
xx = np.linspace(-2.6,2.55)
data.plot(xx,intercept+slope*xx,color="#145c71",label="two uncertain axes")
data.plot(xx,np.polyval(ols,xx),color="#88522f",ls="--",label="vertical fit")
data.set(xlabel="First instrument (mV)",ylabel="Second instrument (K)",title="Where corrections are assigned")
data.legend(fontsize=8)
bb = np.linspace(.75,2.75,351)
cost.plot(bb,[profile(b) for b in bb],color="#145c71")
cost.scatter([slope],[loss],color="#b94626",label="whitened SVD solution")
cost.set(xlabel="Trial sensitivity (K/mV)",ylabel="Sum of squared scaled distances",title="Profile out the unseen settings")
cost.legend(fontsize=8)
for ax in (data,cost): ax.grid(alpha=.2)
plt.show()

## Change units and cross-check the eigensystem
Convert the first instrument to volts and scale its uncertainty too. Transforming the fitted slope back must recover the original. In an installed OneSimulation environment its symmetric eigensystem supplies an independently constructed normal vector.

In [ ]:
factor = np.array([.001,1.])
_,converted,_,converted_loss = orthogonal_fit(P*factor,scales*factor)
assert np.isclose((converted[1]/converted[0])*factor[0],slope)
assert np.isclose(converted_loss,loss)
Z = (P-mean)/scales
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigensystem as eigensystem
except ImportError:
    eigensystem = np.linalg.eigh
ev,_ = eigensystem(Z.T@Z)
assert np.isclose(ev[0],loss)
print("Unit conversion and centred-scatter checks passed.")